
# GraphMS-Net — Stage 13 EDSS / Risk Prediction MAX v1.2 — FINAL VERIFIED

## Goal

Build the strongest **legitimate, reproducible, time-efficient** Stage-13 risk module from the frozen Stage-12 outputs.

### Guide alignment

The guide specifies Stage 13 as a risk-prediction module using:

- **MLP**
- **SVM**
- **XGBoost (optional)**
- lesion-derived features, with deep embeddings when legitimately available.

This notebook therefore evaluates **SVM, MLP and XGBoost** explicitly. It also adds regularized logistic/ridge models and a stabilized linear ensemble because the development cohort is small.

### Scientific design chosen after literature review

1. **Primary classification task:** current high disability, **EDSS ≥ 4**.
   - EDSS 4 marks significant disability while the patient remains independently ambulatory.
   - A 2024 MS machine-learning study reported its best EDSS≥4 discrimination with an SVM combining clinical and global MRI measures (AUC about 0.83 ± 0.07; PMID 38909341).

2. **Secondary task:** continuous EDSS regression.
   - A 2025 systematic review/meta-analysis of MRI-based AI prognosis in MS reported a median classification AUC around 0.78 and median regression RMSE around 1.08 across heterogeneous studies (PMID 40650946). These are context, **not directly comparable targets** for this cohort.

3. **Compact primary predictors rather than 107-variable brute force.**
   - Recent large-cohort MS work identifies age, sex, T2 lesion burden, and regional brain/lesion measures as important contributors to disability.
   - Our primary feature set therefore emphasizes lesion burden, normalized burden, lesion distribution, atlas/periventricular context, age, sex, and disease-course group.
   - The full 107-feature radiomics bank is evaluated only through fold-internal feature selection.

4. **Patient-grouped nested CV.**
   - All timepoints from one patient always stay in the same fold.
   - Hyperparameter tuning occurs only in the inner loop.
   - Outer folds estimate development generalization.
   - This follows the logic of TRIPOD/TRIPOD+AI and nested-CV guidance: preprocessing, feature selection, model tuning, and threshold tuning must not see the outer validation fold.

5. **No test-set model shopping.**
   - The 22 historical-test patients are not loaded until the model lock is written.
   - They are forbidden from feature selection, scaling/imputation fitting, hyperparameter selection, threshold tuning, and model selection.
   - Because this test set has already been exposed elsewhere in the project, its results are labeled **historical holdout, not untouched external validation**.

### Primary model

A compact **MRI/lesion-only regularized linear SVM** is predeclared as the primary classifier. Only its regularization strength is tuned. This follows the guide's lesion-feature risk path and avoids silently making clinical metadata part of the deployable GraphMS risk signal.

Clinical+MRI and clinical-only models are retained as **auxiliary upper-bound comparators**, not as the guide-primary deployment branch.

MRI-only MLP, standalone SVM, XGBoost, full-107-feature MRI models, plus clinical-only and clinical+MRI upper-bound branches are evaluated as comparators.

### Deep-embedding rule

A compatible **ResEncM-250 OOF embedding bank is not currently available**. This notebook will never substitute CATMIL/GAT embeddings or fabricate CNN embeddings. A strictly validated optional hook is included; if a compatible ResEncM-250 embedding bank is later placed at the declared path, it can be added under a new locked protocol.

---

### Key references informing the protocol

- *Evaluation of machine learning-based classification of clinical impairment and prediction of clinical worsening in multiple sclerosis* — 2024, PMID **38909341**.
- *AI-powered disease progression prediction in multiple sclerosis using magnetic resonance imaging: a systematic review and meta-analysis* — 2025, PMID **40650946**.
- *Machine learning for multiple sclerosis classification and disability prediction using clinical and MRI data* — 2026, PMID **42038541**.
- TRIPOD+AI statement — BMJ 2024.
- scikit-learn nested CV and StratifiedGroupKFold documentation.


In [ ]:

# 0. Minimal environment setup.
import importlib.util, subprocess, sys

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "xgboost": "xgboost",
    "matplotlib": "matplotlib",
}
missing = [pip_name for module, pip_name in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages already available.")


In [ ]:

# 1. Mount Drive and freeze Stage-13 configuration.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import os, json, math, time, hashlib, warnings, shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, f_regression
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score,
    roc_curve, precision_recall_curve
)
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path("/content/drive/MyDrive/MSLesSeg_MS")
NNV2 = ROOT / "nnunet_v2"
S12 = NNV2 / "stage12_canonical_features_max_v2"

DEV_PATH = S12 / "stage13_DEV_ONLY_93_crosssectional.csv"
TEST_PATH = S12 / "stage13_HISTORICAL_TEST_ONLY_22_crosssectional.csv"
CONTRACT_PATH = S12 / "STAGE13_RECOMMENDED_FEATURE_COLUMNS.json"
S12_AUDIT = S12 / "STAGE12_FINAL_AUDIT.csv"
S12_MANIFEST = S12 / "STAGE12_CANONICAL_FEATURE_MANIFEST.json"

OUT = NNV2 / "stage13_edss_risk_max_v1"
CKPT_ROOT = OUT / "checkpoints"
FIG_DIR = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

STAGE13_VERSION = "GraphMSNet-Stage13-MAX-v1.2"
IMPLEMENTATION_REVISION = "patient-level-stratification__mri-primary__rev2_final-audit-fix"
EXPECTED_STAGE12_SCHEMA = "GraphMSNet-Stage12-MAX-v2.1"

# Reproducible patient-grouped nested CV.
OUTER_SEEDS = (42, 123, 7)
OUTER_SPLITS = 4
INNER_SPLITS = 3

# Inference / uncertainty settings.
BOOTSTRAPS = 2000
RUN_HISTORICAL_TEST = True
RUN_OPTIONAL_PERMUTATION_TEST = False   # costly and not guide-required
RANDOM_SEED = 42

# Primary task.
CLASS_TARGET = "target_edss_ge4"
REG_TARGET = "target_edss"
PRIMARY_CLASS_FAMILY = "MRI_SPATIAL_SVM"
PRIMARY_REG_FAMILY = "MRI_SPATIAL_RIDGE"

# Optional compatible embeddings only. Nothing is fabricated if absent.
OOF_EMBED_PATH = NNV2 / "stage13_resencm250_embeddings" / "resencm250_oof_embeddings.csv"
TEST_EMBED_PATH = NNV2 / "stage13_resencm250_embeddings" / "resencm250_test_embeddings.csv"

print("Stage-12:", S12)
print("Stage-13 output:", OUT)

PROGRESS_PATH = OUT / "STAGE13_PROGRESS_CHECKPOINT.json"


In [ ]:

# 2. Fail-closed Stage-12 handoff audit.
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

required = [DEV_PATH, TEST_PATH, CONTRACT_PATH, S12_AUDIT, S12_MANIFEST]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing Stage-12 prerequisite(s): {missing}")

contract = json.loads(CONTRACT_PATH.read_text())
if contract.get("feature_schema_version") != EXPECTED_STAGE12_SCHEMA:
    raise RuntimeError(
        f"Stage-12 schema mismatch: {contract.get('feature_schema_version')} != {EXPECTED_STAGE12_SCHEMA}"
    )

dev = pd.read_csv(DEV_PATH)

# IMPORTANT: only header is read here. Historical-test outcomes are not loaded.
test_header = pd.read_csv(TEST_PATH, nrows=0)
TEST_LABELS_LOADED = False

if len(dev) != 93:
    raise RuntimeError(f"Expected 93 development scans, found {len(dev)}")
if dev["patient_id"].nunique() != 53:
    raise RuntimeError(f"Expected 53 development patients, found {dev['patient_id'].nunique()}")
if dev["case_id"].nunique() != 93:
    raise RuntimeError("Development case IDs are not unique.")
if set(dev["role"].unique()) != {"development_oof"}:
    raise RuntimeError(f"Unexpected development roles: {dev['role'].unique().tolist()}")
if dev[REG_TARGET].isna().any() or dev[CLASS_TARGET].isna().any():
    raise RuntimeError("Missing EDSS targets in development data.")

imaging_features = list(contract["recommended_imaging_features"])
missing_features = [c for c in imaging_features if c not in dev.columns]
if missing_features:
    raise RuntimeError(f"Contract imaging features missing from development table: {missing_features[:10]}")

for forbidden in contract.get("qc_only_excluded_from_model", []):
    if forbidden in imaging_features:
        raise RuntimeError(f"QC-only column leaked into feature contract: {forbidden}")

# Test schema compatibility only, without reading rows/labels.
required_test_columns = {"case_id", "patient_id", REG_TARGET, CLASS_TARGET, *imaging_features, "age", "sex", "ms_type"}
missing_test_cols = sorted(required_test_columns - set(test_header.columns))
if missing_test_cols:
    raise RuntimeError(f"Historical-test schema is missing columns: {missing_test_cols[:10]}")

audit = {
    "stage13_version": STAGE13_VERSION,
    "stage12_schema": contract["feature_schema_version"],
    "stage12_dev_sha256": sha256_file(DEV_PATH),
    "stage12_contract_sha256": sha256_file(CONTRACT_PATH),
    "development_scans": int(len(dev)),
    "development_patients": int(dev["patient_id"].nunique()),
    "development_edss_ge4_scans": int(dev[CLASS_TARGET].sum()),
    "development_edss_ge4_patients": int(dev.loc[dev[CLASS_TARGET] == 1, "patient_id"].nunique()),
    "imaging_features": int(len(imaging_features)),
    "historical_test_labels_loaded_before_lock": TEST_LABELS_LOADED,
}
(OUT / "STAGE13_PREREQUISITE_AUDIT.json").write_text(json.dumps(audit, indent=2))

print(json.dumps(audit, indent=2))


In [ ]:

# 3. Deterministic, outcome-blind Stage-13 feature engineering and guide map.

def engineer_stage13_features(df):
    x = df.copy()

    # Dataset codes:
    # SMRR = relapsing-remitting course;
    # SMSP / SMPP = progressive-course variants.
    # Coarsening avoids an unseen progressive subtype becoming an all-zero OHE vector.
    course_map = {
        "SMRR": "RELAPSING",
        "SMSP": "PROGRESSIVE",
        "SMPP": "PROGRESSIVE",
    }
    x["ms_course_group"] = (
        x["ms_type"].astype(str).str.strip().str.upper().map(course_map).fillna("OTHER")
    )

    # Low-dimensional, target-independent nonlinear terms.
    x["age_sq"] = pd.to_numeric(x["age"], errors="coerce") ** 2
    x["age_x_log_lesion_volume"] = (
        pd.to_numeric(x["age"], errors="coerce")
        * pd.to_numeric(x["log_lesion_volume"], errors="coerce")
    )
    return x

dev = engineer_stage13_features(dev)

CLINICAL_CORE = ["age", "sex", "ms_course_group"]

# Compact MS-disability representation:
# burden + normalized burden + lesion distribution/location + brain-volume approximation.
SPATIAL_IMAGING_CORE = [
    "log_lesion_volume",
    "lesion_volume_normalized",
    "lesion_count",
    "largest_lesion_fraction",
    "approx_brain_volume_mm3",
    "atlas_brainstem_overlap_mm3",
    "atlas_cortical_overlap_mm3",
    "atlas_subcortical_overlap_mm3",
    "periventricular_fraction_le3mm",
    "ventricle_distance_mean_mm",
    "lesion_world_x_std_mm",
    "lesion_world_y_std_mm",
    "lesion_world_z_std_mm",
]

ENGINEERED_CORE = ["age_sq", "age_x_log_lesion_volume"]

FEATURE_SETS = {
    "guide_spatial_fused": CLINICAL_CORE + SPATIAL_IMAGING_CORE + ENGINEERED_CORE,
    "mri_spatial": SPATIAL_IMAGING_CORE,
    "clinical_only": CLINICAL_CORE,
    "all107_fused": imaging_features + CLINICAL_CORE + ENGINEERED_CORE,
    "all107_imaging": imaging_features,
}

for name, cols in FEATURE_SETS.items():
    missing = [c for c in cols if c not in dev.columns]
    if missing:
        raise RuntimeError(f"Feature set {name} has missing columns: {missing}")

# Compatible frozen ResEncM embeddings are accepted only if both OOF and test banks exist.
EMBEDDINGS_AVAILABLE = OOF_EMBED_PATH.exists() and TEST_EMBED_PATH.exists()
EMBEDDING_COLUMNS = []

if EMBEDDINGS_AVAILABLE:
    emb = pd.read_csv(OOF_EMBED_PATH)
    if "case_id" not in emb.columns or emb["case_id"].duplicated().any():
        raise RuntimeError("Embedding bank must contain unique case_id.")
    EMBEDDING_COLUMNS = [c for c in emb.columns if c.startswith("emb_")]
    if not EMBEDDING_COLUMNS:
        raise RuntimeError("Compatible embedding bank found but no emb_* columns exist.")
    if set(emb["case_id"]) != set(dev["case_id"]):
        raise RuntimeError("OOF embedding case set does not match 93 development scans.")
    dev = dev.merge(emb[["case_id", *EMBEDDING_COLUMNS]], on="case_id", how="left", validate="one_to_one")
    FEATURE_SETS["mri_spatial_embeddings"] = FEATURE_SETS["mri_spatial"] + EMBEDDING_COLUMNS
    FEATURE_SETS["guide_spatial_fused_embeddings"] = FEATURE_SETS["guide_spatial_fused"] + EMBEDDING_COLUMNS
    print(f"Compatible ResEncM-250 embeddings enabled: {len(EMBEDDING_COLUMNS)} dimensions")
else:
    print("No compatible ResEncM-250 embedding bank present — embeddings will NOT be fabricated or substituted.")

guide_map = pd.DataFrame([
    ["Lesion-derived features", "Required", "Implemented", "Frozen Stage-12 predicted-mask features"],
    ["MLP", "Required/model option", "Implemented comparator", "Small regularized MLP"],
    ["SVM", "Required/model option", "Implemented", "Linear SVM is part of primary ensemble"],
    ["XGBoost", "Optional", "Implemented comparator", "Shallow regularized XGBoost"],
    ["Deep embeddings", "Specified input", "Gated / unavailable now" if not EMBEDDINGS_AVAILABLE else "Implemented", 
     "Only compatible ResEncM-250 OOF/test embeddings are accepted"],
    ["Patient leakage prevention", "Scientific requirement", "Implemented", "Patient-level StratifiedKFold; mapped back to all scans"],
    ["Historical-test isolation", "Scientific requirement", "Implemented", "Test labels loaded only after model lock"],
    ["Clinical metadata", "Auxiliary only", "Upper-bound comparator", "Not part of the guide-primary MRI-only branch"],
], columns=["Guide / scientific item", "Expected", "Status", "Implementation"])

display(guide_map)
guide_map.to_csv(OUT / "STAGE13_GUIDE_ALIGNMENT.csv", index=False)


In [ ]:

# 4. Patient-grouped split, weighting, threshold and bootstrap utilities.

def patient_equal_weights(groups):
    g = pd.Series(np.asarray(groups))
    counts = g.map(g.value_counts()).to_numpy(dtype=float)
    w = 1.0 / counts
    return w / np.mean(w)

def classification_train_weights(y, groups):
    """
    Equalize total contribution of patients, then balance positive/negative class mass.
    """
    y = np.asarray(y, dtype=int)
    base = patient_equal_weights(groups)
    m0 = base[y == 0].sum()
    m1 = base[y == 1].sum()
    if m0 <= 0 or m1 <= 0:
        raise RuntimeError("Training fold lacks one classification class.")
    total = m0 + m1
    class_factor = np.where(y == 1, total / (2 * m1), total / (2 * m0))
    w = base * class_factor
    return w / np.mean(w)

def regression_train_weights(groups):
    return patient_equal_weights(groups)


def patient_classification_table(df):
    """
    One row per patient for fold assignment.
    Stratify by whether the patient is ever EDSS>=4; all of that patient's scans
    are then mapped to the same fold.
    """
    p = (
        df.groupby("patient_id", as_index=False)
        .agg(strat_target=(CLASS_TARGET, "max"), n_scans=("case_id", "size"))
        .sort_values("patient_id")
        .reset_index(drop=True)
    )
    return p

def patient_regression_table(df):
    """
    One row per patient. We stratify on the patient's maximum observed EDSS band
    to distribute severe patients across folds without splitting longitudinal scans.
    """
    p = (
        df.groupby("patient_id", as_index=False)
        .agg(max_edss=(REG_TARGET, "max"), n_scans=("case_id", "size"))
        .sort_values("patient_id")
        .reset_index(drop=True)
    )
    p["strat_target"] = np.asarray(
        pd.cut(
            p["max_edss"].to_numpy(dtype=float),
            bins=[-np.inf, 0.0, 1.5, 3.5, np.inf],
            labels=False,
            include_lowest=True,
        ),
        dtype=int,
    )
    return p

def _map_patient_split_to_rows(df, train_patients, val_patients):
    tr_mask = df["patient_id"].isin(set(train_patients)).to_numpy()
    va_mask = df["patient_id"].isin(set(val_patients)).to_numpy()
    tr = np.flatnonzero(tr_mask)
    va = np.flatnonzero(va_mask)
    if len(set(df.iloc[tr]["patient_id"]) & set(df.iloc[va]["patient_id"])):
        raise RuntimeError("Patient overlap after row mapping.")
    if len(tr) + len(va) != len(df):
        raise RuntimeError("Patient split does not cover every scan exactly once.")
    return tr, va

def classification_splits(df, n_splits, seed):
    p = patient_classification_table(df)
    counts = p["strat_target"].value_counts()
    if int(counts.min()) < n_splits:
        raise RuntimeError(
            f"Not enough positive/negative patients for {n_splits}-fold classification CV: {counts.to_dict()}"
        )

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = []
    for trp, vap in skf.split(p, p["strat_target"]):
        tr_pat = p.iloc[trp]["patient_id"].tolist()
        va_pat = p.iloc[vap]["patient_id"].tolist()
        tr, va = _map_patient_split_to_rows(df, tr_pat, va_pat)

        val_patient_targets = p.iloc[vap]["strat_target"]
        if val_patient_targets.nunique() < 2:
            raise RuntimeError("Classification validation fold lacks a patient-level class.")
        splits.append((tr, va))

    return splits

def regression_splits(df, n_splits, seed):
    p = patient_regression_table(df)
    counts = p["strat_target"].value_counts()
    if int(counts.min()) < n_splits:
        raise RuntimeError(
            f"Not enough patients in an EDSS severity stratum for {n_splits}-fold regression CV: {counts.to_dict()}"
        )

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = []
    for trp, vap in skf.split(p, p["strat_target"]):
        tr_pat = p.iloc[trp]["patient_id"].tolist()
        va_pat = p.iloc[vap]["patient_id"].tolist()
        splits.append(_map_patient_split_to_rows(df, tr_pat, va_pat))
    return splits

def split_audit_table(df):
    rows = []
    for task, fn in [("classification", classification_splits), ("regression", regression_splits)]:
        for seed in OUTER_SEEDS:
            for fold, (tr, va) in enumerate(fn(df, OUTER_SPLITS, seed)):
                val = df.iloc[va]
                rows.append({
                    "task": task,
                    "seed": int(seed),
                    "fold": int(fold),
                    "train_patients": int(df.iloc[tr]["patient_id"].nunique()),
                    "val_patients": int(val["patient_id"].nunique()),
                    "val_scans": int(len(val)),
                    "val_positive_scans": int(val[CLASS_TARGET].sum()) if task == "classification" else np.nan,
                    "val_positive_patients": (
                        int(val.loc[val[CLASS_TARGET] == 1, "patient_id"].nunique())
                        if task == "classification" else np.nan
                    ),
                })
    return pd.DataFrame(rows)

def weighted_auc(y, score, groups):
    return float(roc_auc_score(y, score, sample_weight=patient_equal_weights(groups)))

def weighted_ap(y, score, groups):
    return float(average_precision_score(y, score, sample_weight=patient_equal_weights(groups)))

def classification_metrics(y, prob, pred, groups):
    y = np.asarray(y, dtype=int)
    prob = np.asarray(prob, dtype=float)
    pred = np.asarray(pred, dtype=int)
    w = patient_equal_weights(groups)

    tn = float(np.sum(w[(y == 0) & (pred == 0)]))
    fp = float(np.sum(w[(y == 0) & (pred == 1)]))
    fn = float(np.sum(w[(y == 1) & (pred == 0)]))
    tp = float(np.sum(w[(y == 1) & (pred == 1)]))
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = 2 * precision * sens / (precision + sens) if (precision + sens) else 0.0

    return {
        "roc_auc": float(roc_auc_score(y, prob, sample_weight=w)),
        "average_precision": float(average_precision_score(y, prob, sample_weight=w)),
        "brier": float(np.average((prob - y) ** 2, weights=w)),
        "accuracy": float(np.average((pred == y).astype(float), weights=w)),
        "balanced_accuracy": float((sens + spec) / 2),
        "sensitivity": float(sens),
        "specificity": float(spec),
        "precision": float(precision),
        "f1": float(f1),
    }

def regression_metrics(y, pred, groups):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    w = patient_equal_weights(groups)
    mse = float(np.average((y - pred) ** 2, weights=w))
    mae = float(np.average(np.abs(y - pred), weights=w))
    ybar = float(np.average(y, weights=w))
    denom = float(np.sum(w * (y - ybar) ** 2))
    r2 = 1.0 - float(np.sum(w * (y - pred) ** 2)) / denom if denom > 0 else np.nan
    rho = spearmanr(y, pred).statistic if len(np.unique(y)) > 1 else np.nan
    return {
        "mae": mae,
        "rmse": float(np.sqrt(mse)),
        "r2": float(r2),
        "spearman_rho": float(rho) if np.isfinite(rho) else np.nan,
    }

def tune_thresholds(y, prob, groups, min_high_sensitivity=0.75):
    y = np.asarray(y, dtype=int)
    prob = np.asarray(prob, dtype=float)
    w = patient_equal_weights(groups)

    candidates = np.unique(
        np.r_[0.0, 1.0, np.quantile(prob, np.linspace(0, 1, 401))]
    )

    balanced_rows = []
    high_rows = []
    for t in candidates:
        p = (prob >= t).astype(int)
        m = classification_metrics(y, prob, p, groups)
        balanced_rows.append((m["balanced_accuracy"], m["f1"], m["sensitivity"], m["specificity"], float(t)))
        if m["sensitivity"] >= min_high_sensitivity:
            high_rows.append((m["specificity"], m["precision"], m["f1"], -float(t), float(t)))

    balanced_rows.sort(reverse=True)
    balanced_t = balanced_rows[0][-1]

    if high_rows:
        high_rows.sort(reverse=True)
        high_t = high_rows[0][-1]
    else:
        # fallback: highest sensitivity, then specificity
        all_rows = []
        for t in candidates:
            p = (prob >= t).astype(int)
            m = classification_metrics(y, prob, p, groups)
            all_rows.append((m["sensitivity"], m["specificity"], -float(t), float(t)))
        all_rows.sort(reverse=True)
        high_t = all_rows[0][-1]

    return {
        "balanced_threshold": float(balanced_t),
        "high_sensitivity_threshold": float(high_t),
        "high_sensitivity_target": float(min_high_sensitivity),
    }

def cluster_bootstrap_ci(df, metric_fn, value_columns, n_boot=BOOTSTRAPS, seed=42):
    """
    Resample patients, retaining all rows/timepoints of each sampled patient.
    metric_fn receives a bootstrap DataFrame and returns a dict.
    """
    rng = np.random.default_rng(seed)
    patients = np.array(sorted(df["patient_id"].unique()))
    rows = []

    by_patient = {p: df[df["patient_id"] == p].copy() for p in patients}
    for b in range(n_boot):
        sampled = rng.choice(patients, size=len(patients), replace=True)
        parts = []
        for j, p in enumerate(sampled):
            q = by_patient[p].copy()
            q["_boot_patient"] = f"{p}__{j}"
            parts.append(q)
        boot = pd.concat(parts, ignore_index=True)
        try:
            vals = metric_fn(boot)
            rows.append(vals)
        except Exception:
            continue

    if not rows:
        return {}

    bdf = pd.DataFrame(rows)
    out = {}
    for c in value_columns:
        vals = pd.to_numeric(bdf[c], errors="coerce").dropna()
        if len(vals):
            out[c] = {
                "low": float(vals.quantile(0.025)),
                "high": float(vals.quantile(0.975)),
                "valid_bootstraps": int(len(vals)),
            }
    return out

# Mandatory real-data split audit.
split_audit = split_audit_table(dev)
split_audit.to_csv(OUT / "STAGE13_SPLIT_AUDIT.csv", index=False)

# Every classification outer fold must contain >=2 high-disability patients.
class_audit = split_audit[split_audit["task"] == "classification"]
if (class_audit["val_positive_patients"] < 2).any():
    raise RuntimeError(
        "Patient-level stratification failed: an outer validation fold has <2 EDSS>=4 patients."
    )

# Re-run constructors to catch inner-split feasibility now, before model fitting.
for seed in OUTER_SEEDS:
    for fold, (tr, va) in enumerate(classification_splits(dev, OUTER_SPLITS, seed)):
        train_df = dev.iloc[tr].reset_index(drop=True)
        _ = classification_splits(train_df, INNER_SPLITS, 100000 + int(seed) * 10 + fold)
    for fold, (tr, va) in enumerate(regression_splits(dev, OUTER_SPLITS, seed)):
        train_df = dev.iloc[tr].reset_index(drop=True)
        _ = regression_splits(train_df, INNER_SPLITS, 200000 + int(seed) * 10 + fold)

# Patient weighting audit: every patient's total evaluation weight must match.
w_check = pd.DataFrame({"patient_id": dev["patient_id"], "w": patient_equal_weights(dev["patient_id"])})
totals = w_check.groupby("patient_id")["w"].sum().to_numpy()
assert np.allclose(totals, totals[0], rtol=1e-10, atol=1e-10)

print("CV / weighting self-tests PASSED.")
display(split_audit)


In [ ]:

# 5. Model builders — preprocessing and selection are always fitted inside the training fold.

CATEGORICAL = {"sex", "ms_course_group"}

def make_preprocessor(feature_set_name):
    cols = FEATURE_SETS[feature_set_name]
    numeric = [c for c in cols if c not in CATEGORICAL]
    categorical = [c for c in cols if c in CATEGORICAL]

    transformers = []
    if numeric:
        transformers.append((
            "num",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]),
            numeric,
        ))
    if categorical:
        transformers.append((
            "cat",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical,
        ))

    return ColumnTransformer(transformers, sparse_threshold=0)

def fit_pipeline_weighted(pipe, X, y, sample_weight):
    """
    MLP sample_weight support depends on sklearn version.
    For every other model in this notebook it is supported.
    """
    try:
        pipe.fit(X, y, model__sample_weight=sample_weight)
        return pipe, True
    except TypeError as e:
        if "sample_weight" not in str(e):
            raise
        pipe.fit(X, y)
        return pipe, False

def _class_base_pipeline(feature_set_name, model, k=None):
    steps = [
        ("pre", make_preprocessor(feature_set_name)),
        ("variance", VarianceThreshold(1e-12)),
    ]
    if k is not None:
        steps.append(("select", SelectKBest(f_classif, k=int(k))))
    steps.append(("model", model))
    return Pipeline(steps)

def _reg_base_pipeline(feature_set_name, model, k=None):
    steps = [
        ("pre", make_preprocessor(feature_set_name)),
        ("variance", VarianceThreshold(1e-12)),
    ]
    if k is not None:
        steps.append(("select", SelectKBest(f_regression, k=int(k))))
    steps.append(("model", model))
    return Pipeline(steps)

def _logit_probability(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def fit_class_model(family, params, train_df, feature_set_name):
    cols = FEATURE_SETS[feature_set_name]
    X = train_df[cols]
    y = train_df[CLASS_TARGET].astype(int).to_numpy()
    w = classification_train_weights(y, train_df["patient_id"].to_numpy())

    if family == "ensemble":
        C = float(params["C"])
        log = _class_base_pipeline(
            feature_set_name,
            LogisticRegression(C=C, penalty="l2", solver="liblinear", max_iter=5000, random_state=RANDOM_SEED),
        )
        svm = _class_base_pipeline(
            feature_set_name,
            SVC(C=C, kernel="linear", probability=False, cache_size=500, random_state=RANDOM_SEED),
        )
        log, sw1 = fit_pipeline_weighted(log, X, y, w)
        svm, sw2 = fit_pipeline_weighted(svm, X, y, w)

        log_train = _logit_probability(log.predict_proba(X)[:, 1])
        svm_train = svm.decision_function(X).astype(float)

        return {
            "kind": "ensemble",
            "logistic": log,
            "svm": svm,
            "log_mu": float(np.mean(log_train)),
            "log_sd": float(np.std(log_train) + 1e-8),
            "svm_mu": float(np.mean(svm_train)),
            "svm_sd": float(np.std(svm_train) + 1e-8),
            "sample_weight_supported": bool(sw1 and sw2),
        }

    if family == "logistic":
        model = LogisticRegression(
            C=float(params["C"]), penalty="l2", solver="liblinear", max_iter=5000, random_state=RANDOM_SEED
        )
        pipe = _class_base_pipeline(feature_set_name, model, k=params.get("k"))
    elif family == "linear_svm":
        model = SVC(
            C=float(params["C"]), kernel="linear", probability=False, cache_size=500, random_state=RANDOM_SEED
        )
        pipe = _class_base_pipeline(feature_set_name, model, k=params.get("k"))
    elif family == "xgb":
        model = XGBClassifier(
            n_estimators=int(params["n_estimators"]),
            max_depth=int(params["max_depth"]),
            learning_rate=float(params.get("learning_rate", 0.05)),
            subsample=0.9,
            colsample_bytree=0.8,
            reg_lambda=10.0,
            reg_alpha=0.2,
            min_child_weight=2.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=2,
            verbosity=0,
        )
        pipe = _class_base_pipeline(feature_set_name, model, k=params.get("k"))
    elif family == "mlp":
        model = MLPClassifier(
            hidden_layer_sizes=tuple(params.get("hidden", (16,))),
            alpha=float(params["alpha"]),
            activation="relu",
            solver="lbfgs",
            max_iter=1000,
            random_state=RANDOM_SEED,
        )
        pipe = _class_base_pipeline(feature_set_name, model, k=params.get("k"))
    else:
        raise ValueError(f"Unknown classification family: {family}")

    pipe, supported = fit_pipeline_weighted(pipe, X, y, w)
    return {"kind": family, "pipeline": pipe, "sample_weight_supported": bool(supported)}

def predict_class_raw(bundle, df, feature_set_name):
    X = df[FEATURE_SETS[feature_set_name]]

    if bundle["kind"] == "ensemble":
        log = bundle["logistic"]
        svm = bundle["svm"]
        a = _logit_probability(log.predict_proba(X)[:, 1])
        b = svm.decision_function(X).astype(float)
        za = (a - bundle["log_mu"]) / bundle["log_sd"]
        zb = (b - bundle["svm_mu"]) / bundle["svm_sd"]
        return (za + zb) / 2.0

    pipe = bundle["pipeline"]
    if hasattr(pipe, "decision_function"):
        return np.asarray(pipe.decision_function(X), dtype=float)
    p = pipe.predict_proba(X)[:, 1]
    return _logit_probability(p)

def fit_reg_model(family, params, train_df, feature_set_name):
    cols = FEATURE_SETS[feature_set_name]
    X = train_df[cols]
    y = train_df[REG_TARGET].astype(float).to_numpy()
    w = regression_train_weights(train_df["patient_id"].to_numpy())

    if family == "ridge":
        model = Ridge(alpha=float(params["alpha"]))
        pipe = _reg_base_pipeline(feature_set_name, model, k=params.get("k"))
    elif family == "linear_svr":
        model = SVR(
            C=float(params["C"]),
            kernel="linear",
            epsilon=float(params.get("epsilon", 0.5)),
            cache_size=500,
        )
        pipe = _reg_base_pipeline(feature_set_name, model, k=params.get("k"))
    elif family == "xgb":
        model = XGBRegressor(
            n_estimators=int(params["n_estimators"]),
            max_depth=int(params["max_depth"]),
            learning_rate=float(params.get("learning_rate", 0.05)),
            subsample=0.9,
            colsample_bytree=0.8,
            reg_lambda=10.0,
            reg_alpha=0.2,
            min_child_weight=2.0,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=2,
            verbosity=0,
        )
        pipe = _reg_base_pipeline(feature_set_name, model, k=params.get("k"))
    elif family == "mlp":
        model = MLPRegressor(
            hidden_layer_sizes=tuple(params.get("hidden", (16,))),
            alpha=float(params["alpha"]),
            activation="relu",
            solver="lbfgs",
            max_iter=1000,
            random_state=RANDOM_SEED,
        )
        pipe = _reg_base_pipeline(feature_set_name, model, k=params.get("k"))
    else:
        raise ValueError(f"Unknown regression family: {family}")

    pipe, supported = fit_pipeline_weighted(pipe, X, y, w)
    return {"kind": family, "pipeline": pipe, "sample_weight_supported": bool(supported)}

def predict_reg(bundle, df, feature_set_name):
    pred = np.asarray(
        bundle["pipeline"].predict(df[FEATURE_SETS[feature_set_name]]),
        dtype=float
    )
    return np.clip(pred, 0.0, 10.0)

print("Model builders ready.")


In [ ]:

# 6. Predeclared compact model registry.
# GUIDE-PRIMARY = lesion/MRI features only.
# Clinical+MRI and clinical-only branches are explicitly auxiliary upper-bound comparators.
# No family is promoted after looking at the historical test set.

CLASS_FAMILIES = [
    {
        "name": "MRI_SPATIAL_SVM",
        "family": "linear_svm",
        "feature_set": "mri_spatial",
        "grid": [{"C": c} for c in (1.0, 3.0, 10.0, 30.0, 100.0)],
        "role": "PRIMARY_GUIDE_ALIGNED",
    },
    {
        "name": "MRI_SPATIAL_LOGISTIC",
        "family": "logistic",
        "feature_set": "mri_spatial",
        "grid": [{"C": c} for c in (0.1, 0.3, 1.0, 3.0, 10.0)],
        "role": "STABILITY_COMPARATOR",
    },
    {
        "name": "MRI_SPATIAL_XGB",
        "family": "xgb",
        "feature_set": "mri_spatial",
        "grid": [
            {"max_depth": d, "n_estimators": n, "learning_rate": 0.05}
            for d in (1, 2) for n in (100, 200)
        ],
        "role": "GUIDE_COMPARATOR",
    },
    {
        "name": "MRI_SPATIAL_MLP",
        "family": "mlp",
        "feature_set": "mri_spatial",
        "grid": [{"hidden": (16,), "alpha": a} for a in (0.1, 1.0, 10.0)],
        "role": "GUIDE_COMPARATOR",
    },
    {
        "name": "ALL107_MRI_SVM",
        "family": "linear_svm",
        "feature_set": "all107_imaging",
        "grid": [
            {"C": c, "k": k}
            for k in (16, 32, 64)
            for c in (1.0, 10.0, 30.0)
        ],
        "role": "HIGH_DIMENSIONAL_MRI_COMPARATOR",
    },
    {
        "name": "FUSED_CLINICAL_MRI_SVM",
        "family": "linear_svm",
        "feature_set": "guide_spatial_fused",
        "grid": [{"C": c} for c in (1.0, 3.0, 10.0, 30.0)],
        "role": "CLINICAL_UPPER_BOUND",
    },
    {
        "name": "CLINICAL_ONLY_SVM",
        "family": "linear_svm",
        "feature_set": "clinical_only",
        "grid": [{"C": c} for c in (1.0, 3.0, 10.0, 30.0)],
        "role": "CLINICAL_ONLY_COMPARATOR",
    },
]

REG_FAMILIES = [
    {
        "name": "MRI_SPATIAL_RIDGE",
        "family": "ridge",
        "feature_set": "mri_spatial",
        "grid": [{"alpha": a} for a in (0.3, 1.0, 3.0, 10.0, 30.0)],
        "role": "PRIMARY_GUIDE_ALIGNED",
    },
    {
        "name": "MRI_SPATIAL_SVR",
        "family": "linear_svr",
        "feature_set": "mri_spatial",
        "grid": [{"C": c, "epsilon": 0.5} for c in (0.03, 0.1, 0.3, 1.0, 3.0)],
        "role": "GUIDE_COMPARATOR",
    },
    {
        "name": "MRI_SPATIAL_XGB_REG",
        "family": "xgb",
        "feature_set": "mri_spatial",
        "grid": [
            {"max_depth": d, "n_estimators": n, "learning_rate": 0.05}
            for d in (1, 2) for n in (100, 200)
        ],
        "role": "GUIDE_COMPARATOR",
    },
    {
        "name": "MRI_SPATIAL_MLP_REG",
        "family": "mlp",
        "feature_set": "mri_spatial",
        "grid": [{"hidden": (16,), "alpha": a} for a in (0.1, 1.0, 10.0)],
        "role": "GUIDE_COMPARATOR",
    },
    {
        "name": "ALL107_MRI_RIDGE",
        "family": "ridge",
        "feature_set": "all107_imaging",
        "grid": [
            {"alpha": a, "k": k}
            for k in (16, 32, 64)
            for a in (3.0, 10.0, 30.0)
        ],
        "role": "HIGH_DIMENSIONAL_MRI_COMPARATOR",
    },
    {
        "name": "FUSED_CLINICAL_MRI_RIDGE",
        "family": "ridge",
        "feature_set": "guide_spatial_fused",
        "grid": [{"alpha": a} for a in (0.3, 1.0, 3.0, 10.0, 30.0)],
        "role": "CLINICAL_UPPER_BOUND",
    },
    {
        "name": "CLINICAL_ONLY_RIDGE",
        "family": "ridge",
        "feature_set": "clinical_only",
        "grid": [{"alpha": a} for a in (0.3, 1.0, 3.0, 10.0, 30.0)],
        "role": "CLINICAL_ONLY_COMPARATOR",
    },
]

# If a real frozen ResEncM-250 embedding bank appears, evaluate it without changing
# the MRI-only primary result after the historical test has been opened.
if EMBEDDINGS_AVAILABLE:
    CLASS_FAMILIES.append({
        "name": "MRI_SPATIAL_PLUS_RESENCM_EMB_SVM",
        "family": "linear_svm",
        "feature_set": "mri_spatial_embeddings",
        "grid": [{"C": c} for c in (0.3, 1.0, 3.0, 10.0)],
        "role": "GUIDE_DEEP_EMBEDDING_COMPARATOR",
    })
    REG_FAMILIES.append({
        "name": "MRI_SPATIAL_PLUS_RESENCM_EMB_RIDGE",
        "family": "ridge",
        "feature_set": "mri_spatial_embeddings",
        "grid": [{"alpha": a} for a in (1.0, 10.0, 100.0)],
        "role": "GUIDE_DEEP_EMBEDDING_COMPARATOR",
    })

registry_payload = {
    "stage13_version": STAGE13_VERSION,
    "implementation_revision": IMPLEMENTATION_REVISION,
    "split_policy": "patient-level stratification then scan mapping",
    "outer_seeds": list(OUTER_SEEDS),
    "outer_splits": OUTER_SPLITS,
    "inner_splits": INNER_SPLITS,
    "class_families": CLASS_FAMILIES,
    "reg_families": REG_FAMILIES,
    "feature_sets": FEATURE_SETS,
    "stage12_dev_sha256": sha256_file(DEV_PATH),
    "stage12_contract_sha256": sha256_file(CONTRACT_PATH),
}

RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(registry_payload, sort_keys=True, default=list, separators=(",", ":")).encode()
).hexdigest()
CKPT = CKPT_ROOT / RUN_FINGERPRINT[:16]
CKPT.mkdir(parents=True, exist_ok=True)

(OUT / "STAGE13_MODEL_REGISTRY.json").write_text(
    json.dumps(registry_payload, indent=2, default=list)
)

print("Run fingerprint:", RUN_FINGERPRINT)
print("Checkpoint folder:", CKPT)
print("Classification families:", len(CLASS_FAMILIES))
print("Regression families:", len(REG_FAMILIES))


In [ ]:

# 7. Nested-CV engine with fold-level checkpointing.

def _json_params(p):
    return json.dumps(p, sort_keys=True, separators=(",", ":"), default=list)

def _atomic_json(path, obj):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=float))
    os.replace(tmp, path)


def write_stage13_progress(task, family, seed=None, fold=None, status="RUNNING"):
    existing = {}
    if PROGRESS_PATH.exists():
        try:
            existing = json.loads(PROGRESS_PATH.read_text())
        except Exception:
            existing = {}
    completed = len(list(CKPT.glob("*.json")))
    payload = {
        "stage13_version": STAGE13_VERSION,
        "implementation_revision": IMPLEMENTATION_REVISION,
        "run_fingerprint": RUN_FINGERPRINT,
        "status": status,
        "current_task": task,
        "current_family": family,
        "current_seed": seed,
        "current_fold": fold,
        "completed_fold_checkpoints": int(completed),
        "updated_utc": pd.Timestamp.utcnow().isoformat(),
    }
    _atomic_json(PROGRESS_PATH, payload)

def _inner_class_param_score(fdef, params, train_df, inner_splits):
    aucs, aps = [], []
    for tr, va in inner_splits:
        a = train_df.iloc[tr].reset_index(drop=True)
        b = train_df.iloc[va].reset_index(drop=True)
        bundle = fit_class_model(fdef["family"], params, a, fdef["feature_set"])
        raw = predict_class_raw(bundle, b, fdef["feature_set"])
        aucs.append(weighted_auc(b[CLASS_TARGET].astype(int), raw, b["patient_id"]))
        aps.append(weighted_ap(b[CLASS_TARGET].astype(int), raw, b["patient_id"]))
    return float(np.mean(aucs)), float(np.mean(aps))

def _inner_reg_param_score(fdef, params, train_df, inner_splits):
    rmses, maes = [], []
    for tr, va in inner_splits:
        a = train_df.iloc[tr].reset_index(drop=True)
        b = train_df.iloc[va].reset_index(drop=True)
        bundle = fit_reg_model(fdef["family"], params, a, fdef["feature_set"])
        pred = predict_reg(bundle, b, fdef["feature_set"])
        m = regression_metrics(
            b[REG_TARGET].to_numpy(),
            pred,
            b["patient_id"].to_numpy(),
        )
        rmses.append(m["rmse"])
        maes.append(m["mae"])
    return float(np.mean(rmses)), float(np.mean(maes))

def run_nested_class_family(fdef):
    summaries, predictions, leaderboards = [], [], []

    for seed in OUTER_SEEDS:
        outer = classification_splits(dev, OUTER_SPLITS, seed)

        for fold, (tr, va) in enumerate(outer):
            ck = CKPT / f"class_{fdef['name']}_seed{seed}_fold{fold}.json"

            write_stage13_progress("classification", fdef["name"], int(seed), int(fold), "RUNNING")

            if ck.exists():
                obj = json.loads(ck.read_text())
                if obj.get("run_fingerprint") != RUN_FINGERPRINT:
                    raise RuntimeError(f"Stale checkpoint fingerprint: {ck}")
                summaries.append(obj["summary"])
                predictions.extend(obj["predictions"])
                leaderboards.extend(obj["leaderboard"])
                print("RESTORED", ck.name)
                continue

            train_df = dev.iloc[tr].reset_index(drop=True)
            val_df = dev.iloc[va].reset_index(drop=True)

            inner = classification_splits(
                train_df,
                INNER_SPLITS,
                seed=100000 + int(seed) * 10 + fold,
            )

            lb = []
            for j, params in enumerate(fdef["grid"], start=1):
                try:
                    auc, ap = _inner_class_param_score(fdef, params, train_df, inner)
                    status = "ok"
                except Exception as e:
                    auc, ap = np.nan, np.nan
                    status = repr(e)

                lb.append({
                    "task": "classification",
                    "family_name": fdef["name"],
                    "outer_seed": int(seed),
                    "outer_fold": int(fold),
                    "params": params,
                    "param_key": _json_params(params),
                    "inner_auc": None if not np.isfinite(auc) else float(auc),
                    "inner_ap": None if not np.isfinite(ap) else float(ap),
                    "status": status,
                })

            valid = [r for r in lb if r["inner_auc"] is not None]
            if not valid:
                raise RuntimeError(f"All candidate params failed for {fdef['name']} seed={seed} fold={fold}")

            # Primary discrimination: AUC. AP is the deterministic tie-breaker.
            valid.sort(key=lambda r: (r["inner_auc"], r["inner_ap"]), reverse=True)
            best = valid[0]
            best_params = best["params"]

            bundle = fit_class_model(fdef["family"], best_params, train_df, fdef["feature_set"])
            raw = predict_class_raw(bundle, val_df, fdef["feature_set"])

            outer_auc = weighted_auc(
                val_df[CLASS_TARGET].astype(int),
                raw,
                val_df["patient_id"],
            )
            outer_ap = weighted_ap(
                val_df[CLASS_TARGET].astype(int),
                raw,
                val_df["patient_id"],
            )

            summary = {
                "task": "classification",
                "family_name": fdef["name"],
                "role": fdef["role"],
                "feature_set": fdef["feature_set"],
                "outer_seed": int(seed),
                "outer_fold": int(fold),
                "n_val": int(len(val_df)),
                "n_val_patients": int(val_df["patient_id"].nunique()),
                "n_val_positive": int(val_df[CLASS_TARGET].sum()),
                "best_params": best_params,
                "best_inner_auc": float(best["inner_auc"]),
                "best_inner_ap": float(best["inner_ap"]),
                "outer_auc": float(outer_auc),
                "outer_ap": float(outer_ap),
            }

            fold_preds = []
            for row_i, raw_i in zip(range(len(val_df)), raw):
                fold_preds.append({
                    "task": "classification",
                    "family_name": fdef["name"],
                    "outer_seed": int(seed),
                    "outer_fold": int(fold),
                    "case_id": val_df.iloc[row_i]["case_id"],
                    "patient_id": val_df.iloc[row_i]["patient_id"],
                    "y_true": int(val_df.iloc[row_i][CLASS_TARGET]),
                    "raw_score": float(raw_i),
                })

            obj = {
                "run_fingerprint": RUN_FINGERPRINT,
                "summary": summary,
                "predictions": fold_preds,
                "leaderboard": lb,
            }
            _atomic_json(ck, obj)

            summaries.append(summary)
            predictions.extend(fold_preds)
            leaderboards.extend(lb)

            print(
                f"{fdef['name']} seed={seed} fold={fold} "
                f"innerAUC={best['inner_auc']:.3f} outerAUC={outer_auc:.3f}"
            )

    return (
        pd.DataFrame(summaries),
        pd.DataFrame(predictions),
        pd.DataFrame(leaderboards),
    )

def run_nested_reg_family(fdef):
    summaries, predictions, leaderboards = [], [], []

    for seed in OUTER_SEEDS:
        outer = regression_splits(dev, OUTER_SPLITS, seed)

        for fold, (tr, va) in enumerate(outer):
            ck = CKPT / f"reg_{fdef['name']}_seed{seed}_fold{fold}.json"

            write_stage13_progress("regression", fdef["name"], int(seed), int(fold), "RUNNING")

            if ck.exists():
                obj = json.loads(ck.read_text())
                if obj.get("run_fingerprint") != RUN_FINGERPRINT:
                    raise RuntimeError(f"Stale checkpoint fingerprint: {ck}")
                summaries.append(obj["summary"])
                predictions.extend(obj["predictions"])
                leaderboards.extend(obj["leaderboard"])
                print("RESTORED", ck.name)
                continue

            train_df = dev.iloc[tr].reset_index(drop=True)
            val_df = dev.iloc[va].reset_index(drop=True)

            inner = regression_splits(
                train_df,
                INNER_SPLITS,
                seed=200000 + int(seed) * 10 + fold,
            )

            lb = []
            for params in fdef["grid"]:
                try:
                    rmse, mae = _inner_reg_param_score(fdef, params, train_df, inner)
                    status = "ok"
                except Exception as e:
                    rmse, mae = np.nan, np.nan
                    status = repr(e)

                lb.append({
                    "task": "regression",
                    "family_name": fdef["name"],
                    "outer_seed": int(seed),
                    "outer_fold": int(fold),
                    "params": params,
                    "param_key": _json_params(params),
                    "inner_rmse": None if not np.isfinite(rmse) else float(rmse),
                    "inner_mae": None if not np.isfinite(mae) else float(mae),
                    "status": status,
                })

            valid = [r for r in lb if r["inner_rmse"] is not None]
            if not valid:
                raise RuntimeError(f"All regression params failed for {fdef['name']} seed={seed} fold={fold}")

            # Primary regression objective: RMSE. MAE is deterministic tie-breaker.
            valid.sort(key=lambda r: (r["inner_rmse"], r["inner_mae"]))
            best = valid[0]
            best_params = best["params"]

            bundle = fit_reg_model(fdef["family"], best_params, train_df, fdef["feature_set"])
            pred = predict_reg(bundle, val_df, fdef["feature_set"])

            met = regression_metrics(
                val_df[REG_TARGET].to_numpy(),
                pred,
                val_df["patient_id"].to_numpy(),
            )

            summary = {
                "task": "regression",
                "family_name": fdef["name"],
                "role": fdef["role"],
                "feature_set": fdef["feature_set"],
                "outer_seed": int(seed),
                "outer_fold": int(fold),
                "n_val": int(len(val_df)),
                "n_val_patients": int(val_df["patient_id"].nunique()),
                "best_params": best_params,
                "best_inner_rmse": float(best["inner_rmse"]),
                "best_inner_mae": float(best["inner_mae"]),
                **{f"outer_{k}": float(v) for k, v in met.items()},
            }

            fold_preds = []
            for row_i, pred_i in zip(range(len(val_df)), pred):
                fold_preds.append({
                    "task": "regression",
                    "family_name": fdef["name"],
                    "outer_seed": int(seed),
                    "outer_fold": int(fold),
                    "case_id": val_df.iloc[row_i]["case_id"],
                    "patient_id": val_df.iloc[row_i]["patient_id"],
                    "y_true": float(val_df.iloc[row_i][REG_TARGET]),
                    "prediction": float(pred_i),
                })

            obj = {
                "run_fingerprint": RUN_FINGERPRINT,
                "summary": summary,
                "predictions": fold_preds,
                "leaderboard": lb,
            }
            _atomic_json(ck, obj)

            summaries.append(summary)
            predictions.extend(fold_preds)
            leaderboards.extend(lb)

            print(
                f"{fdef['name']} seed={seed} fold={fold} "
                f"innerRMSE={best['inner_rmse']:.3f} outerRMSE={met['rmse']:.3f}"
            )

    return (
        pd.DataFrame(summaries),
        pd.DataFrame(predictions),
        pd.DataFrame(leaderboards),
    )

print("Nested-CV engine ready.")


In [ ]:

# 8. Classification nested CV — MRI-primary + guide/comparator families.
class_summary_parts = []
class_pred_parts = []
class_lb_parts = []

t0 = time.perf_counter()
for fdef in CLASS_FAMILIES:
    print("\n==", fdef["name"], "==")
    s, p, l = run_nested_class_family(fdef)
    class_summary_parts.append(s)
    class_pred_parts.append(p)
    class_lb_parts.append(l)

class_outer = pd.concat(class_summary_parts, ignore_index=True)
class_outer_predictions = pd.concat(class_pred_parts, ignore_index=True)
class_leaderboard = pd.concat(class_lb_parts, ignore_index=True)

class_family_summary = (
    class_outer.groupby(["family_name", "role", "feature_set"], as_index=False)
    .agg(
        nested_auc_mean=("outer_auc", "mean"),
        nested_auc_std=("outer_auc", "std"),
        nested_auc_median=("outer_auc", "median"),
        nested_ap_mean=("outer_ap", "mean"),
        nested_ap_std=("outer_ap", "std"),
        folds=("outer_auc", "size"),
    )
    .sort_values("nested_auc_mean", ascending=False)
)

class_outer.to_csv(OUT / "stage13_classification_nested_outer_folds.csv", index=False)
class_outer_predictions.to_csv(OUT / "stage13_classification_nested_outer_predictions.csv", index=False)
class_leaderboard.to_csv(OUT / "stage13_classification_inner_leaderboards.csv", index=False)
class_family_summary.to_csv(OUT / "stage13_classification_family_summary.csv", index=False)

print(f"\nClassification nested CV wall time: {(time.perf_counter()-t0)/60:.2f} min")
display(class_family_summary)


In [ ]:

# 9. Lock the primary classifier using development data only.
primary_def = next(x for x in CLASS_FAMILIES if x["name"] == PRIMARY_CLASS_FAMILY)

primary_lb = class_leaderboard[
    (class_leaderboard["family_name"] == PRIMARY_CLASS_FAMILY)
    & (class_leaderboard["status"] == "ok")
].copy()

if primary_lb.empty:
    raise RuntimeError("Primary classification family has no valid inner-CV results.")

consensus = (
    primary_lb.groupby("param_key", as_index=False)
    .agg(
        mean_inner_auc=("inner_auc", "mean"),
        mean_inner_ap=("inner_ap", "mean"),
        evaluations=("inner_auc", "size"),
    )
    .sort_values(["mean_inner_auc", "mean_inner_ap"], ascending=False)
)

best_param_key = consensus.iloc[0]["param_key"]
PRIMARY_CLASS_PARAMS = json.loads(best_param_key)

print("Locked MRI-only primary classification params:", PRIMARY_CLASS_PARAMS)
display(consensus)

# Fixed-spec repeated grouped OOF predictions.
fixed_rows = []
for seed in OUTER_SEEDS:
    for fold, (tr, va) in enumerate(classification_splits(dev, OUTER_SPLITS, seed)):
        train_df = dev.iloc[tr].reset_index(drop=True)
        val_df = dev.iloc[va].reset_index(drop=True)
        bundle = fit_class_model(
            primary_def["family"],
            PRIMARY_CLASS_PARAMS,
            train_df,
            primary_def["feature_set"],
        )
        raw = predict_class_raw(bundle, val_df, primary_def["feature_set"])
        for i, s in enumerate(raw):
            fixed_rows.append({
                "case_id": val_df.iloc[i]["case_id"],
                "patient_id": val_df.iloc[i]["patient_id"],
                "y_true": int(val_df.iloc[i][CLASS_TARGET]),
                "seed": int(seed),
                "fold": int(fold),
                "raw_score": float(s),
            })

fixed_class = pd.DataFrame(fixed_rows)
fixed_class.to_csv(OUT / "stage13_primary_class_fixed_oof_repeats.csv", index=False)

agg = (
    fixed_class.groupby(["case_id", "patient_id", "y_true"], as_index=False)
    .agg(raw_score=("raw_score", "mean"))
)

if len(agg) != 93 or agg["case_id"].nunique() != 93:
    raise RuntimeError("Fixed classification OOF aggregation is incomplete.")

# OOF Platt calibration. It never sees historical-test outcomes.
cal_w = patient_equal_weights(agg["patient_id"])
platt = LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000, random_state=RANDOM_SEED)
platt.fit(agg[["raw_score"]].to_numpy(), agg["y_true"].to_numpy(), sample_weight=cal_w)
agg["probability"] = platt.predict_proba(agg[["raw_score"]].to_numpy())[:, 1]

thresholds = tune_thresholds(
    agg["y_true"].to_numpy(),
    agg["probability"].to_numpy(),
    agg["patient_id"].to_numpy(),
    min_high_sensitivity=0.75,
)

agg["pred_balanced"] = (agg["probability"] >= thresholds["balanced_threshold"]).astype(int)
agg["pred_high_sensitivity"] = (
    agg["probability"] >= thresholds["high_sensitivity_threshold"]
).astype(int)

dev_balanced_metrics = classification_metrics(
    agg["y_true"], agg["probability"], agg["pred_balanced"], agg["patient_id"]
)
dev_high_metrics = classification_metrics(
    agg["y_true"], agg["probability"], agg["pred_high_sensitivity"], agg["patient_id"]
)

# Fit deployable classifier on all 93 development scans.
final_class_model = fit_class_model(
    primary_def["family"],
    PRIMARY_CLASS_PARAMS,
    dev.reset_index(drop=True),
    primary_def["feature_set"],
)

class_bundle = {
    "stage13_version": STAGE13_VERSION,
    "run_fingerprint": RUN_FINGERPRINT,
    "task": "EDSS>=4 classification",
    "primary_family": PRIMARY_CLASS_FAMILY,
    "family": primary_def["family"],
    "feature_set": primary_def["feature_set"],
    "feature_columns": FEATURE_SETS[primary_def["feature_set"]],
    "params": PRIMARY_CLASS_PARAMS,
    "model": final_class_model,
    "calibrator": platt,
    "thresholds": thresholds,
    "development_fixed_oof_balanced_metrics": dev_balanced_metrics,
    "development_fixed_oof_high_sensitivity_metrics": dev_high_metrics,
    "nested_family_performance": class_family_summary[
        class_family_summary["family_name"] == PRIMARY_CLASS_FAMILY
    ].to_dict("records")[0],
    "historical_test_used_for_model_selection": False,
}

CLASS_MODEL_PATH = OUT / "stage13_primary_edss_ge4_classifier.joblib"
joblib.dump(class_bundle, CLASS_MODEL_PATH)

class_lock_payload = {
    "run_fingerprint": RUN_FINGERPRINT,
    "family": PRIMARY_CLASS_FAMILY,
    "params": PRIMARY_CLASS_PARAMS,
    "feature_set": primary_def["feature_set"],
    "thresholds": thresholds,
    "platt_coef": float(platt.coef_.ravel()[0]),
    "platt_intercept": float(platt.intercept_.ravel()[0]),
    "dev_sha256": sha256_file(DEV_PATH),
}
CLASS_LOCK_ID = hashlib.sha256(
    json.dumps(class_lock_payload, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()
class_lock_payload["lock_id"] = CLASS_LOCK_ID

(OUT / "STAGE13_CLASSIFICATION_MODEL_LOCK.json").write_text(
    json.dumps(class_lock_payload, indent=2)
)
agg.to_csv(OUT / "stage13_primary_class_fixed_oof_aggregated.csv", index=False)

print("Classification lock:", CLASS_LOCK_ID)
print("Thresholds:", thresholds)
print("Development fixed-OOF balanced-threshold metrics:")
print(json.dumps(dev_balanced_metrics, indent=2))
print("Development fixed-OOF high-sensitivity metrics:")
print(json.dumps(dev_high_metrics, indent=2))


In [ ]:

# 10. Regression nested CV — continuous EDSS.
reg_summary_parts = []
reg_pred_parts = []
reg_lb_parts = []

t0 = time.perf_counter()
for fdef in REG_FAMILIES:
    print("\n==", fdef["name"], "==")
    s, p, l = run_nested_reg_family(fdef)
    reg_summary_parts.append(s)
    reg_pred_parts.append(p)
    reg_lb_parts.append(l)

reg_outer = pd.concat(reg_summary_parts, ignore_index=True)
reg_outer_predictions = pd.concat(reg_pred_parts, ignore_index=True)
reg_leaderboard = pd.concat(reg_lb_parts, ignore_index=True)

reg_family_summary = (
    reg_outer.groupby(["family_name", "role", "feature_set"], as_index=False)
    .agg(
        nested_rmse_mean=("outer_rmse", "mean"),
        nested_rmse_std=("outer_rmse", "std"),
        nested_rmse_median=("outer_rmse", "median"),
        nested_mae_mean=("outer_mae", "mean"),
        nested_mae_std=("outer_mae", "std"),
        nested_r2_mean=("outer_r2", "mean"),
        nested_spearman_mean=("outer_spearman_rho", "mean"),
        folds=("outer_rmse", "size"),
    )
    .sort_values("nested_rmse_mean")
)

reg_outer.to_csv(OUT / "stage13_regression_nested_outer_folds.csv", index=False)
reg_outer_predictions.to_csv(OUT / "stage13_regression_nested_outer_predictions.csv", index=False)
reg_leaderboard.to_csv(OUT / "stage13_regression_inner_leaderboards.csv", index=False)
reg_family_summary.to_csv(OUT / "stage13_regression_family_summary.csv", index=False)

print(f"\nRegression nested CV wall time: {(time.perf_counter()-t0)/60:.2f} min")
display(reg_family_summary)


In [ ]:

# 11. Lock the primary EDSS regressor using development data only.
primary_reg_def = next(x for x in REG_FAMILIES if x["name"] == PRIMARY_REG_FAMILY)

primary_reg_lb = reg_leaderboard[
    (reg_leaderboard["family_name"] == PRIMARY_REG_FAMILY)
    & (reg_leaderboard["status"] == "ok")
].copy()

if primary_reg_lb.empty:
    raise RuntimeError("Primary regression family has no valid inner-CV results.")

reg_consensus = (
    primary_reg_lb.groupby("param_key", as_index=False)
    .agg(
        mean_inner_rmse=("inner_rmse", "mean"),
        mean_inner_mae=("inner_mae", "mean"),
        evaluations=("inner_rmse", "size"),
    )
    .sort_values(["mean_inner_rmse", "mean_inner_mae"])
)

PRIMARY_REG_PARAMS = json.loads(reg_consensus.iloc[0]["param_key"])
print("Locked MRI-only primary regression params:", PRIMARY_REG_PARAMS)
display(reg_consensus)

fixed_reg_rows = []
for seed in OUTER_SEEDS:
    for fold, (tr, va) in enumerate(regression_splits(dev, OUTER_SPLITS, seed)):
        train_df = dev.iloc[tr].reset_index(drop=True)
        val_df = dev.iloc[va].reset_index(drop=True)
        bundle = fit_reg_model(
            primary_reg_def["family"],
            PRIMARY_REG_PARAMS,
            train_df,
            primary_reg_def["feature_set"],
        )
        pred = predict_reg(bundle, val_df, primary_reg_def["feature_set"])

        for i, q in enumerate(pred):
            fixed_reg_rows.append({
                "case_id": val_df.iloc[i]["case_id"],
                "patient_id": val_df.iloc[i]["patient_id"],
                "y_true": float(val_df.iloc[i][REG_TARGET]),
                "seed": int(seed),
                "fold": int(fold),
                "prediction": float(q),
            })

fixed_reg = pd.DataFrame(fixed_reg_rows)
fixed_reg.to_csv(OUT / "stage13_primary_reg_fixed_oof_repeats.csv", index=False)

reg_agg = (
    fixed_reg.groupby(["case_id", "patient_id", "y_true"], as_index=False)
    .agg(prediction=("prediction", "mean"))
)
if len(reg_agg) != 93:
    raise RuntimeError("Fixed regression OOF aggregation is incomplete.")

dev_reg_metrics = regression_metrics(
    reg_agg["y_true"],
    reg_agg["prediction"],
    reg_agg["patient_id"],
)

final_reg_model = fit_reg_model(
    primary_reg_def["family"],
    PRIMARY_REG_PARAMS,
    dev.reset_index(drop=True),
    primary_reg_def["feature_set"],
)

reg_bundle = {
    "stage13_version": STAGE13_VERSION,
    "run_fingerprint": RUN_FINGERPRINT,
    "task": "continuous EDSS regression",
    "primary_family": PRIMARY_REG_FAMILY,
    "family": primary_reg_def["family"],
    "feature_set": primary_reg_def["feature_set"],
    "feature_columns": FEATURE_SETS[primary_reg_def["feature_set"]],
    "params": PRIMARY_REG_PARAMS,
    "model": final_reg_model,
    "development_fixed_oof_metrics": dev_reg_metrics,
    "nested_family_performance": reg_family_summary[
        reg_family_summary["family_name"] == PRIMARY_REG_FAMILY
    ].to_dict("records")[0],
    "historical_test_used_for_model_selection": False,
}

REG_MODEL_PATH = OUT / "stage13_primary_edss_regressor.joblib"
joblib.dump(reg_bundle, REG_MODEL_PATH)

reg_lock_payload = {
    "run_fingerprint": RUN_FINGERPRINT,
    "family": PRIMARY_REG_FAMILY,
    "params": PRIMARY_REG_PARAMS,
    "feature_set": primary_reg_def["feature_set"],
    "dev_sha256": sha256_file(DEV_PATH),
}
REG_LOCK_ID = hashlib.sha256(
    json.dumps(reg_lock_payload, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()
reg_lock_payload["lock_id"] = REG_LOCK_ID

(OUT / "STAGE13_REGRESSION_MODEL_LOCK.json").write_text(
    json.dumps(reg_lock_payload, indent=2)
)
reg_agg.to_csv(OUT / "stage13_primary_reg_fixed_oof_aggregated.csv", index=False)

print("Regression lock:", REG_LOCK_ID)
print("Development fixed-OOF regression metrics:")
print(json.dumps(dev_reg_metrics, indent=2))


In [ ]:

# 12. Development plots + confidence intervals for the locked primary models.

def _dev_class_boot_metric(b):
    return classification_metrics(
        b["y_true"].to_numpy(),
        b["probability"].to_numpy(),
        b["pred_balanced"].to_numpy(),
        b["_boot_patient"].to_numpy(),
    )

def _dev_reg_boot_metric(b):
    return regression_metrics(
        b["y_true"].to_numpy(),
        b["prediction"].to_numpy(),
        b["_boot_patient"].to_numpy(),
    )

dev_class_ci = cluster_bootstrap_ci(
    agg,
    _dev_class_boot_metric,
    ["roc_auc", "average_precision", "balanced_accuracy", "sensitivity", "specificity", "f1"],
    n_boot=BOOTSTRAPS,
    seed=42,
)
dev_reg_ci = cluster_bootstrap_ci(
    reg_agg,
    _dev_reg_boot_metric,
    ["mae", "rmse", "r2", "spearman_rho"],
    n_boot=BOOTSTRAPS,
    seed=43,
)

(OUT / "STAGE13_DEVELOPMENT_BOOTSTRAP_CI.json").write_text(
    json.dumps({"classification": dev_class_ci, "regression": dev_reg_ci}, indent=2)
)

# ROC
fpr, tpr, _ = roc_curve(agg["y_true"], agg["probability"])
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC={dev_balanced_metrics['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], "--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Stage 13 development fixed-OOF ROC")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "development_fixed_oof_roc.png", dpi=180)
plt.show()

# PR
pr, rc, _ = precision_recall_curve(agg["y_true"], agg["probability"])
plt.figure(figsize=(6, 5))
plt.plot(rc, pr, label=f"AP={dev_balanced_metrics['average_precision']:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Stage 13 development fixed-OOF Precision–Recall")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "development_fixed_oof_pr.png", dpi=180)
plt.show()

# Regression
plt.figure(figsize=(6, 5))
plt.scatter(reg_agg["y_true"], reg_agg["prediction"], alpha=0.75)
lims = [0, max(8, float(reg_agg["y_true"].max()) + 0.5)]
plt.plot(lims, lims, "--", linewidth=1)
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel("Observed EDSS")
plt.ylabel("OOF predicted EDSS")
plt.title("Stage 13 development fixed-OOF EDSS regression")
plt.tight_layout()
plt.savefig(FIG_DIR / "development_fixed_oof_regression.png", dpi=180)
plt.show()

print("Development classification CI:")
print(json.dumps(dev_class_ci, indent=2))
print("Development regression CI:")
print(json.dumps(dev_reg_ci, indent=2))


In [ ]:

# 13. Locked historical-test evaluation.
# This cell is the FIRST point at which historical-test labels are loaded.

MODEL_LOCK_PAYLOAD = {
    "classification_lock_id": CLASS_LOCK_ID,
    "regression_lock_id": REG_LOCK_ID,
    "run_fingerprint": RUN_FINGERPRINT,
}
MODEL_LOCK_ID = hashlib.sha256(
    json.dumps(MODEL_LOCK_PAYLOAD, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

TEST_LOCK_PATH = OUT / "STAGE13_HISTORICAL_TEST_LOCK.json"
TEST_METRICS_PATH = OUT / "STAGE13_HISTORICAL_TEST_METRICS.json"
TEST_PRED_PATH = OUT / "stage13_historical_test_predictions.csv"

historical_test_result = None

if RUN_HISTORICAL_TEST:
    if TEST_LOCK_PATH.exists():
        prior = json.loads(TEST_LOCK_PATH.read_text())
        if prior.get("model_lock_id") != MODEL_LOCK_ID:
            raise RuntimeError(
                "Historical test was previously evaluated under a DIFFERENT model lock. "
                "Refusing to re-open the test set after model changes."
            )
        if TEST_METRICS_PATH.exists() and TEST_PRED_PATH.exists():
            historical_test_result = json.loads(TEST_METRICS_PATH.read_text())
            print("Historical-test result restored under identical model lock.")
        else:
            raise RuntimeError("Test lock exists but test artifacts are incomplete.")
    else:
        # First and only new test evaluation for this model lock.
        test = engineer_stage13_features(pd.read_csv(TEST_PATH))
        TEST_LABELS_LOADED = True

        if len(test) != 22 or test["patient_id"].nunique() != 22:
            raise RuntimeError(
                f"Expected 22 one-scan historical-test patients; got rows={len(test)}, "
                f"patients={test['patient_id'].nunique()}"
            )
        overlap = set(dev["patient_id"]) & set(test["patient_id"])
        if overlap:
            raise RuntimeError(f"Development/historical-test patient overlap: {sorted(overlap)}")

        # If compatible embeddings were enabled, test embeddings are joined now only.
        if EMBEDDINGS_AVAILABLE:
            temb = pd.read_csv(TEST_EMBED_PATH)
            if "case_id" not in temb.columns or temb["case_id"].duplicated().any():
                raise RuntimeError("Test embedding bank has invalid case_id.")
            if set(temb["case_id"]) != set(test["case_id"]):
                raise RuntimeError("Test embedding case set mismatch.")
            test = test.merge(
                temb[["case_id", *EMBEDDING_COLUMNS]],
                on="case_id",
                how="left",
                validate="one_to_one",
            )

        # Classification
        raw_test = predict_class_raw(
            final_class_model,
            test,
            primary_def["feature_set"],
        )
        prob_test = platt.predict_proba(raw_test.reshape(-1, 1))[:, 1]

        pred_bal = (prob_test >= thresholds["balanced_threshold"]).astype(int)
        pred_hi = (prob_test >= thresholds["high_sensitivity_threshold"]).astype(int)

        test_class_bal = classification_metrics(
            test[CLASS_TARGET].astype(int),
            prob_test,
            pred_bal,
            test["patient_id"],
        )
        test_class_hi = classification_metrics(
            test[CLASS_TARGET].astype(int),
            prob_test,
            pred_hi,
            test["patient_id"],
        )

        # Regression
        pred_edss = predict_reg(
            final_reg_model,
            test,
            primary_reg_def["feature_set"],
        )
        test_reg = regression_metrics(
            test[REG_TARGET],
            pred_edss,
            test["patient_id"],
        )

        test_pred = pd.DataFrame({
            "case_id": test["case_id"],
            "patient_id": test["patient_id"],
            "true_edss": test[REG_TARGET].astype(float),
            "true_edss_ge4": test[CLASS_TARGET].astype(int),
            "risk_probability": prob_test,
            "risk_pred_balanced": pred_bal,
            "risk_pred_high_sensitivity": pred_hi,
            "predicted_edss": pred_edss,
        })

        def _test_class_boot(b):
            return classification_metrics(
                b["true_edss_ge4"].to_numpy(),
                b["risk_probability"].to_numpy(),
                b["risk_pred_balanced"].to_numpy(),
                b["_boot_patient"].to_numpy(),
            )

        def _test_reg_boot(b):
            return regression_metrics(
                b["true_edss"].to_numpy(),
                b["predicted_edss"].to_numpy(),
                b["_boot_patient"].to_numpy(),
            )

        test_class_ci = cluster_bootstrap_ci(
            test_pred,
            _test_class_boot,
            ["roc_auc", "average_precision", "balanced_accuracy", "sensitivity", "specificity", "f1"],
            n_boot=BOOTSTRAPS,
            seed=44,
        )
        test_reg_ci = cluster_bootstrap_ci(
            test_pred,
            _test_reg_boot,
            ["mae", "rmse", "r2", "spearman_rho"],
            n_boot=BOOTSTRAPS,
            seed=45,
        )

        historical_test_result = {
            "model_lock_id": MODEL_LOCK_ID,
            "historical_test_not_blind_project_level": True,
            "historical_test_used_for_model_selection": False,
            "n_test": int(len(test)),
            "n_positive_edss_ge4": int(test[CLASS_TARGET].sum()),
            "classification_balanced_threshold": test_class_bal,
            "classification_high_sensitivity_threshold": test_class_hi,
            "regression": test_reg,
            "bootstrap_95ci": {
                "classification_balanced_threshold": test_class_ci,
                "regression": test_reg_ci,
            },
        }

        test_pred.to_csv(TEST_PRED_PATH, index=False)
        TEST_METRICS_PATH.write_text(json.dumps(historical_test_result, indent=2))

        _atomic_json(TEST_LOCK_PATH, {
            "model_lock_id": MODEL_LOCK_ID,
            "classification_lock_id": CLASS_LOCK_ID,
            "regression_lock_id": REG_LOCK_ID,
            "run_fingerprint": RUN_FINGERPRINT,
            "historical_test_not_blind_project_level": True,
            "used_for_model_selection": False,
            "created_utc": pd.Timestamp.utcnow().isoformat(),
        })

        print("Historical test evaluated under locked model.")
        print(json.dumps(historical_test_result, indent=2))
else:
    print("Historical-test evaluation disabled by configuration.")


In [ ]:

# 14. Final Stage-13 audit, manifest and package.

audit_rows = []

def gate(name, ok, detail):
    audit_rows.append({"gate": name, "pass": bool(ok), "detail": str(detail)})

primary_class_nested = class_family_summary[
    class_family_summary["family_name"] == PRIMARY_CLASS_FAMILY
].iloc[0].to_dict()

primary_reg_nested = reg_family_summary[
    reg_family_summary["family_name"] == PRIMARY_REG_FAMILY
].iloc[0].to_dict()

gate("Stage12 schema v2.1", contract["feature_schema_version"] == EXPECTED_STAGE12_SCHEMA, contract["feature_schema_version"])
gate("93 development scans", len(dev) == 93, len(dev))
gate("53 development patients", dev["patient_id"].nunique() == 53, dev["patient_id"].nunique())
gate("Patient-grouped outer CV", len(class_outer) == len(CLASS_FAMILIES) * len(OUTER_SEEDS) * OUTER_SPLITS, len(class_outer))
gate("Primary classification family completed", PRIMARY_CLASS_FAMILY in set(class_family_summary["family_name"]), PRIMARY_CLASS_FAMILY)
gate("Primary regression family completed", PRIMARY_REG_FAMILY in set(reg_family_summary["family_name"]), PRIMARY_REG_FAMILY)
gate("SVM guide-primary completed", "MRI_SPATIAL_SVM" in set(class_family_summary["family_name"]), "MRI_SPATIAL_SVM")
gate("MLP comparator completed", "MRI_SPATIAL_MLP" in set(class_family_summary["family_name"]), "MRI_SPATIAL_MLP")
gate("XGBoost comparator completed", "MRI_SPATIAL_XGB" in set(class_family_summary["family_name"]), "MRI_SPATIAL_XGB")
gate("Classifier artifact exists", CLASS_MODEL_PATH.exists(), CLASS_MODEL_PATH)
gate("Regressor artifact exists", REG_MODEL_PATH.exists(), REG_MODEL_PATH)
gate("Classification lock exists", (OUT / "STAGE13_CLASSIFICATION_MODEL_LOCK.json").exists(), CLASS_LOCK_ID)
gate("Regression lock exists", (OUT / "STAGE13_REGRESSION_MODEL_LOCK.json").exists(), REG_LOCK_ID)
gate("Historical test forbidden from selection", True, "loaded only after model locks")
gate("No fabricated deep embeddings", True, f"available={EMBEDDINGS_AVAILABLE}")
gate(
    "Primary classifier is MRI/lesion-only",
    set(FEATURE_SETS[primary_def["feature_set"]]).isdisjoint({"age", "sex", "ms_course_group", "ms_type"}),
    primary_def["feature_set"],
)
gate(
    "Primary regressor is MRI/lesion-only",
    set(FEATURE_SETS[primary_reg_def["feature_set"]]).isdisjoint({"age", "sex", "ms_course_group", "ms_type"}),
    primary_reg_def["feature_set"],
)
gate(
    "Primary nested AUC finite",
    np.isfinite(float(primary_class_nested["nested_auc_mean"])),
    primary_class_nested["nested_auc_mean"],
)
gate(
    "Primary nested RMSE finite",
    np.isfinite(float(primary_reg_nested["nested_rmse_mean"])),
    primary_reg_nested["nested_rmse_mean"],
)

if RUN_HISTORICAL_TEST:
    gate("Historical test lock exists", TEST_LOCK_PATH.exists(), TEST_LOCK_PATH)
    gate("Historical test metrics exist", TEST_METRICS_PATH.exists(), TEST_METRICS_PATH)
    if historical_test_result is not None:
        gate(
            "Historical-test classification AUC finite",
            np.isfinite(float(historical_test_result["classification_balanced_threshold"]["roc_auc"])),
            historical_test_result["classification_balanced_threshold"]["roc_auc"],
        )
        gate(
            "Historical-test regression RMSE finite",
            np.isfinite(float(historical_test_result["regression"]["rmse"])),
            historical_test_result["regression"]["rmse"],
        )

final_audit = pd.DataFrame(audit_rows)
final_audit.to_csv(OUT / "STAGE13_FINAL_AUDIT.csv", index=False)
display(final_audit)

failed = final_audit.loc[~final_audit["pass"]]
if len(failed):
    raise RuntimeError(f"STAGE13 FINAL AUDIT FAILED: {failed['gate'].tolist()}")

manifest = {
    "status": "COMPLETE",
    "stage": "13. Risk Prediction Module",
    "stage13_version": STAGE13_VERSION,
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "run_fingerprint": RUN_FINGERPRINT,
    "stage12_schema": contract["feature_schema_version"],
    "development": {
        "scans": 93,
        "patients": 53,
        "edss_ge4_scans": int(dev[CLASS_TARGET].sum()),
        "edss_ge4_patients": int(dev.loc[dev[CLASS_TARGET] == 1, "patient_id"].nunique()),
    },
    "protocol": {
        "outer_cv": f"{len(OUTER_SEEDS)} repeats x {OUTER_SPLITS}-fold patient-level StratifiedKFold, mapped to scans",
        "inner_cv": f"{INNER_SPLITS}-fold patient-level StratifiedKFold, mapped to scans",
        "group": "patient_id",
        "patient_balanced_evaluation": True,
        "primary_feature_policy": "MRI/lesion-only; clinical variables are comparator-only",
        "patient_and_class_balanced_training_weights": True,
        "feature_selection_inside_training_folds": True,
        "threshold_tuning_from development OOF only": True,
        "historical_test_used_for_model_selection": False,
    },
    "guide_alignment": {
        "lesion_features": True,
        "svm": True,
        "mlp": True,
        "xgboost": True,
        "deep_embeddings": bool(EMBEDDINGS_AVAILABLE),
        "deep_embedding_note": (
            "Compatible ResEncM-250 OOF/test embeddings used."
            if EMBEDDINGS_AVAILABLE
            else "No compatible frozen ResEncM-250 bank exists; embeddings were not fabricated or substituted."
        ),
    },
    "primary_classification": {
        "family": PRIMARY_CLASS_FAMILY,
        "params": PRIMARY_CLASS_PARAMS,
        "lock_id": CLASS_LOCK_ID,
        "nested_cv": primary_class_nested,
        "development_fixed_oof_balanced": dev_balanced_metrics,
        "development_fixed_oof_high_sensitivity": dev_high_metrics,
        "thresholds": thresholds,
        "artifact": str(CLASS_MODEL_PATH),
    },
    "primary_regression": {
        "family": PRIMARY_REG_FAMILY,
        "params": PRIMARY_REG_PARAMS,
        "lock_id": REG_LOCK_ID,
        "nested_cv": primary_reg_nested,
        "development_fixed_oof": dev_reg_metrics,
        "artifact": str(REG_MODEL_PATH),
    },
    "historical_test": historical_test_result,
    "historical_test_note": (
        "This 22-case split is historical holdout evidence, not untouched external validation."
    ),
}

MANIFEST_PATH = OUT / "STAGE13_FINAL_MANIFEST.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, default=float))

zip_base = NNV2 / "GraphMSNet_Stage13_EDSS_Risk_MAX_v1"
zip_path = Path(str(zip_base) + ".zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_base), "zip", root_dir=str(OUT))

write_stage13_progress("complete", "all", status="COMPLETE")
print("\nSTAGE 13 COMPLETE — all hard gates passed.")
print("Manifest:", MANIFEST_PATH)
print("Classifier:", CLASS_MODEL_PATH)
print("Regressor:", REG_MODEL_PATH)
print("Package:", zip_path)



## Stage-13 completion rule

Proceed only if the final cell prints:

**`STAGE 13 COMPLETE — all hard gates passed.`**

### Interpretation discipline

- The **nested-CV mean ± SD** of the MRI/lesion-only primary is the main guide-aligned development estimate because it includes inner hyperparameter tuning.
- The fixed-spec OOF table is for calibration, threshold locking, plots and deployment preparation.
- The 22-case evaluation is a **historical holdout**, not an untouched external cohort.
- Do not replace the primary model with a comparator merely because one comparator happens to score higher on the historical test.
- If a compatible ResEncM-250 deep-embedding bank is later produced, it requires a **new Stage-13 protocol/version and new model lock** rather than modifying this result after seeing test performance.

### Next

Stage 14 should document the guide's multi-task formulation without reopening the frozen segmentation training unless there is a compelling new scientific reason. Stage 15 then packages the frozen segmentation + Stage-12 features + these locked Stage-13 artifacts into the unseen-patient inference workflow.


### Checkpointing

- Every completed outer fold is atomically checkpointed under a run fingerprint that includes:
  - Stage-12 development-table hash
  - Stage-12 feature-contract hash
  - model registry
  - split policy
  - Stage-13 implementation revision
- `STAGE13_PROGRESS_CHECKPOINT.json` records current progress.
- A restart restores only checkpoints from the exact same run fingerprint.
